# Initialiser client Boto 3

In [ ]:
import os
import io 
import boto3
import json
from pprint import pprint
import pandas as pd
from tqdm.notebook import tqdm  #  Barre Jupyter native (bleue)
# ou : from tqdm.autonotebook import tqdm  # auto console/notebook

import pyarrow.parquet as pq
import pyarrow.json as paj
import pyarrow as pa

from tqdm import tqdm
import time

endpoint = os.environ["S3_ENDPOINT_URL"]
bucket = os.environ["S3_BUCKET"]

s3_boto = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)


# Tools

In [ ]:
def read_jsonl_from_s3(s3_client, bucket: str, key: str):
    response = s3_client.get_object(Bucket=bucket, Key=key)
    body = response["Body"].read().decode("utf-8")

    records = []
    for line in body.splitlines():
        if line.strip():
            records.append(json.loads(line))

    return records

def read_jsonl_stream(bucket: str, key: str):
    response = client.get_object(Bucket=bucket, Key=key)
    for line in response["Body"].iter_lines():
        if line:
            yield json.loads(line)

import pyarrow.json as paj
import pyarrow as pa
import io

def read_jsonl_arrow(bucket, key):
    obj = client.get_object(Bucket=bucket, Key=key)
    data = obj["Body"].read()
    table = paj.read_json(io.BytesIO(data))
    return table.to_pandas()

import os
import boto3
import pandas as pd

def get_df_from_s3_jsonl( client, Prefixe):
    
    resp = client.list_objects_v2(
        Bucket=S3_BUCKET,
        Prefix = Prefixe
    )
    
    keys = [obj["Key"] for obj in resp.get("Contents", [])]
    
    dfs = []
    
    for key in keys:
        #print(f"key : {key}")
        if key.endswith(".jsonl"):
            df_part = read_jsonl_arrow(S3_BUCKET, key)
            dfs.append(df_part)
    
    df = pd.concat(dfs, ignore_index=True)
    return df


def list_object_from_key(s3_client, bucket, key, limit=5 ):
    resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=key)
    ret = [obj["Key"] for obj in resp.get("Contents", [])]
    if(limit!=0) :
        return ret[:limit]
    else :
        return ret

def get_all_df_from_s3_jsonl(s3_client, bucket: str, prefix: str = "") -> pd.DataFrame:
    all_records = []
    all_errors = []
    # 1. Liste JSONL
    jsonl_keys = []
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            if obj['Key'].endswith('.jsonl'):
                jsonl_keys.append(obj['Key'])
    
    if not jsonl_keys:
        print("⚠️ Aucun JSONL")
        return pd.DataFrame()
    
    print(f"📂 {len(jsonl_keys)} JSONL...")
    
    # 2. tqdm.notebook → barre Jupyter !
    pbar = tqdm(jsonl_keys, desc="JSONL", unit="fichiers", 
               bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]")
    
    for key in pbar:
        try:
            response = s3_client.get_object(Bucket=bucket, Key=key)
            lines = response['Body'].read().decode('utf-8').splitlines()
            
            n_new = 0
            for line in lines:
                line = line.strip()
                if line:
                    all_records.append(json.loads(line))
                    n_new += 1
            
            pbar.set_postfix({"nouveau": f"{n_new:,}"})
            
        except Exception as e:
            #all_errors.apend( (f"❌ {key.split('/')[-1]}: {e}") )
           print(f"❌ {key}: {e}")

    
    # 3. DataFrame
    df = pd.DataFrame(all_records)
    print(f"✓ {len(all_records):,} records | {df.shape}")

    if all_errors:
        print(f"⚠️ {len(all_errors)} erreurs:")
        for err in all_errors[:50]:  # top 50
            print(f"  • {err}")
        if len(all_errors) > 50:
            print(f"  ... et {len(all_errors)-50} autres")
    return df

# Lister des objets Boto3

In [ ]:
list_object_from_key( s3_boto, bucket, "france_travail/bronze/",5)

# Lire un JSONL (bronze) via boto3

In [ ]:
key = "france_travail/bronze/offers/dt=2026-02-13/run_id=20260213T082635Z/code_rome=A1101/segment=global/part-000001.jsonl"
df_all_records = get_all_df_from_s3_jsonl(s3_boto, bucket, key) 
print(f"Total records: {len(df_all_records)}")


# Lire un lot de JSONL (bronze) via boto3

In [ ]:
#key="france_travail/bronze/offers/"
key="france_travail/bronze/offers/dt=2026-02-16/run_id=20260216T211818Z"
df_all_records = get_all_df_from_s3_jsonl(s3_boto, bucket, key) 
print(f"Total records: {len(df_all_records)}")

# Identification pb Jsonl

In [ ]:
def debug_jsonl_line(s3_client, bucket, key, max_lines=10):
    """Trouve ligne + contexte JSON corrompu"""
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    lines = obj["Body"].read().decode("utf-8").splitlines()
    
    print(f"📄 {key}: {len(lines)} lignes")
    
    for i, line in enumerate(lines[:max_lines]):
        try:
            json.loads(line)
        except json.JSONDecodeError as e:
            print(f"❌ Ligne {i+1}: {e}")
            print(f"   Aperçu: {line[:1793]}...")
            print(f"   Colonne {e.colno}: {' ' * (e.colno-1)}^")
            return i+1
    print("✅ Toutes OK (top 10)")
    return None

# Usage
debug_jsonl_line(s3_boto, bucket, "france_travail/bronze/offers/dt=2026-02-13/run_id=20260213T082635Z/code_rome=D1209/segment=global/part-000001.jsonl")


In [ ]:
import re
def debug_invisible_json(s3_client, bucket, key):
    """Trouve caractères cachés JSON"""
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    content = obj["Body"].read().decode("utf-8")
    
    # 1. Test encodages alternatifs
    for encoding in ["utf-8", "utf-8-sig", "latin1", "cp1252"]:
        try:
            decoded = content.encode().decode(encoding)
            json.loads(decoded)
            print(f"✅ OK avec {encoding}")
            return decoded
        except:
            pass
    
    # 2. Cherche caractères suspects autour col 1793
    pos = 1792
    snippet = content[max(0,pos-100):pos+100]
    print(f"🔍 Autour char 1792 ({len(snippet)}c):")
    print(repr(snippet))  # montre \x00, \u2028 etc.
    
    # 3. Nettoie + retry
    clean = re.sub(r'[\x00-\x1F\x7F-\x9F\u2028\u2029]', '', content)
    try:
        json.loads(clean)
        print("✅ Nettoyé OK")
    except Exception as e:
        print(f"❌ Encore erreur: {e}")

debug_invisible_json(s3_boto, bucket, "france_travail/bronze/offers/dt=2026-02-13/run_id=20260213T082635Z/code_rome=D1209/segment=global/part-000001.jsonl")


In [ ]:
def debug_jsonl_structure(s3_client, bucket, key):
    """Vérifie structure JSONL"""
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    content = obj["Body"].read().decode("utf-8")
    
    lines = content.splitlines()
    print(f"📄 {len(lines)} lignes après splitlines()")
    
    # Test premières lignes
    for i in range(min(150, len(lines))):
        line = lines[i].strip()
        if line:
            try:
                data = json.loads(line)
                print(f"✅ Ligne {i}: {data.get('romeCode', '?')} - {len(str(data))}c")
            except json.JSONDecodeError as e:
                print(f"❌ Ligne {i}: {e}")
                print(f"   Début: {repr(line[:100])}...")
                print(f"   Fin: {repr(line[-100:])}")

debug_jsonl_structure(s3_boto, bucket, "france_travail/bronze/offers/dt=2026-02-13/run_id=20260213T082635Z/code_rome=D1209/segment=global/part-000001.jsonl")


# Chargement du référentiel ROME

In [ ]:
df_referential_rome = get_all_df_from_s3_jsonl(s3_boto, bucket, "france_travail/bronze/rome/")
print("Total rows:", len(df_referential_rome))
df_referential_rome.head()


# Identification pb sur les jsonl

In [ ]:
df_rome_count = df_all_records.groupby("romeCode").size().reset_index(name="n_offres")

df_rome_count = df_rome_count.merge(
    df_referential_rome,
    left_on="romeCode",
    right_on="code",
    how="left"
)

df_rome_count = df_rome_count.drop(columns=["code"])
df_rome_count = df_rome_count[
    ["romeCode", "libelle", "n_offres"]
]
df_rome_count.sort_values("n_offres", ascending=False).head(50)


In [ ]:
# Top X
def left(s: str, n: int) -> str:
    """Tronque à n caractères"""
    return s[:n] + "..." if len(s) > n else s

top_x_value = 80
top_x = df_rome_count.nlargest(top_x_value, "n_offres")

plt.figure(figsize=(12, 12))
bars = plt.barh(range(len(top_x)), top_x["n_offres"], color='steelblue', alpha=0.8)
plt.yticks(range(len(top_x)), 
           [f"{left(libelle,20)} ({code}) " for code, n, libelle in zip(top_x["romeCode"], top_x["n_offres"], top_x["libelle"] )], fontsize=8) 

plt.xlabel("Nombre d'offres", fontsize=12)
plt.title(f"Top {top_x_value} Codes ROME par nombre d'offres")
plt.gca().invert_yaxis()  # top en haut
plt.grid(axis='x', alpha=0.3)

# Valeurs sur barres
for i, (bar, n) in enumerate(zip(bars, top_x["n_offres"])):
    plt.text(n + 50, i, f'{n:,}', va='center', fontsize=6)

plt.tight_layout()
plt.show()


